## Nasa (05 tập)

In [26]:
# ============================================================
# Software Fault Prediction - NASA PROMISE (FINAL CLEAN)
# 6 Metaheuristics + 6 Models + Full Metrics + No Warning
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings

# ===== TẮT WARNING =====
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from mealpy import FloatVar, BBOA as MealpyBBOA, ZOA as MealpyZOA

# ==========================
# ELM
# ==========================
class ELM:
    def __init__(self, input_size, hidden_size=50):
        self.input_size = input_size
        self.hidden_size = hidden_size

    def _act(self, x):
        return 1/(1+np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)
        H = self._act(X @ self.W + self.b)
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        y = H @ self.beta
        return (y > 0.5).astype(int)

    def predict_proba(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta


# ============================================================
# 🔥 LOAD DATA (FIX ALL NASA ISSUES)
# ============================================================
def load_data(path):
    df = pd.read_csv(path).dropna()

    y = df.iloc[:, -1]

    # label xử lý
    if y.dtype == 'object':
        y = y.astype(str).str.lower().str.strip()
        if set(y.unique()).issubset({'yes', 'no'}):
            y = y.map({'yes': 1, 'no': 0})
        else:
            y = pd.factorize(y)[0]
    else:
        y = (y > 0).astype(int)

    X = df.iloc[:, :-1]
    X = X.select_dtypes(include=[np.number]).values

    return X, y.values


# ==========================
# Fitness
# ==========================

def fitness(X, y, sol):
    # chọn top-6 features
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    # ===== scoring (không dùng ML) =====
    scores = np.mean(X_sel, axis=1)

    # normalize về [0,1]
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    # threshold
    preds = (scores >= 0.5).astype(int)

    # ===== confusion matrix =====
    TP = np.sum((y == 1) & (preds == 1))
    TN = np.sum((y == 0) & (preds == 0))
    FP = np.sum((y == 0) & (preds == 1))
    FN = np.sum((y == 1) & (preds == 0))

    # ===== tránh chia 0 =====
    recall = TP / (TP + FN + 1e-9)
    specificity = TN / (TN + FP + 1e-9)

    return recall + specificity

def fitness1(X, y, sol):
    idx = np.argsort(sol)[-6:]
    X_sel = X[:, idx]

    model = MLPClassifier(hidden_layer_sizes=(30,), max_iter=200)
    model.fit(X_sel, y)
    pred = model.predict(X_sel)

    return f1_score(y, pred, average='weighted')

def fitness_fs(X, y, sol):
    """KNN 3-fold CV F1 on top-6 features (wrapper FS for MEALPY BBOA/ZOA)."""
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    idx = np.argsort(sol)[-6:]
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    return cross_val_score(
        KNeighborsClassifier(n_neighbors=5), X[:, idx], y, cv=cv, scoring="f1_weighted"
    ).mean()


# ==========================
# Metaheuristics
# ==========================
def init_pop(n, d): return np.random.rand(n, d)

def GA(f, d, it=20, n=10):
    P = init_pop(n, d)
    for _ in range(it):
        P = np.array(sorted(P, key=f)[-n//2:])
        C = []
        for _ in range(n//2):
            i, j = np.random.randint(len(P)), np.random.randint(len(P))
            cp = np.random.randint(d)
            C.append(np.concatenate([P[i][:cp], P[j][cp:]]))
        P = np.vstack((P, C))
        P += np.random.normal(0, 0.1, P.shape)
        P = np.clip(P, 0, 1)
    return max(P, key=f)

def PSO(f, d, it=20, n=10):
    X = init_pop(n, d); V = np.zeros_like(X); p = X.copy()
    for _ in range(it):
        for i in range(n):
            if f(X[i]) > f(p[i]): p[i] = X[i]
        g = max(p, key=f)
        for i in range(n):
            r1, r2 = np.random.rand(d), np.random.rand(d)
            V[i] = 0.5*V[i] + 1.5*r1*(p[i]-X[i]) + 1.5*r2*(g-X[i])
            X[i] = np.clip(X[i] + V[i], 0, 1)
    return g

def GWO(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        X = sorted(X, key=f)
        a = 2 - 2*t/it
        for i in range(n):
            for leader in X[-3:]:
                A = 2*a*np.random.rand(d)-a
                C = 2*np.random.rand(d)
                D = abs(C*leader - X[i])
                X[i] = leader - A*D
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def Squirrel(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        best = max(X, key=f)
        for i in range(n):
            X[i] += np.random.rand(d)*(best - X[i])
        X = np.clip(X, 0, 1)
    return best

def MRA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for _ in range(it):
        fit = np.array([f(x) for x in X])
        probs = fit / (fit.sum()+1e-9)
        newX = []
        for _ in range(n):
            i = np.random.choice(range(n), p=probs)
            newX.append(X[i] + np.random.normal(0, 0.1, d))
        X = np.clip(np.array(newX), 0, 1)
    return max(X, key=f)

def SSA(f, d, it=20, n=10):
    X = init_pop(n, d)
    for t in range(it):
        c1 = 2*np.exp(-(4*t/it)**2)
        food = max(X, key=f)
        for i in range(n):
            if i == 0:
                X[i] = food + c1*(np.random.rand(d)-0.5)
            else:
                X[i] = (X[i] + X[i-1]) / 2
        X = np.clip(X, 0, 1)
    return max(X, key=f)

def init_pop1(n, d, lb, ub):
    return lb + (ub - lb) * np.random.rand(n, d)

FS_EPOCH = 100
FS_POP_SIZE = 30
FS_LB, FS_UB = 0.0, 1.0
BBOA_SEED = 42
ZOA_SEED = 2024

def run_mealpy_optimizer(optimizer_cls, f, d, epoch=FS_EPOCH, pop_size=FS_POP_SIZE,
                         lb=FS_LB, ub=FS_UB, seed=None):
    """Feature-weight search via MEALPY (maximization on [lb, ub]^d)."""
    problem = {
        "obj_func": f,
        "bounds": FloatVar(lb=(lb,) * d, ub=(ub,) * d),
        "minmax": "max",
        "log_to": None,
    }
    model = optimizer_cls(epoch=epoch, pop_size=pop_size, seed=seed)
    return model.solve(problem).solution

def BBOA(f, d, it=FS_EPOCH, n=FS_POP_SIZE, lb=FS_LB, ub=FS_UB):
    """Brown-Bear Optimization (MEALPY OriginalBBOA)."""
    return run_mealpy_optimizer(MealpyBBOA.OriginalBBOA, f, d, it, n, lb, ub, seed=BBOA_SEED)

def ZOA(f, d, it=FS_EPOCH, n=FS_POP_SIZE, lb=FS_LB, ub=FS_UB):
    """Zebra Optimization Algorithm (MEALPY OriginalZOA)."""
    return run_mealpy_optimizer(MealpyZOA.OriginalZOA, f, d, it, n, lb, ub, seed=ZOA_SEED)


def SSO(f, d, it=100, n=30, lb=0, ub=1):
    X = init_pop1(n, d, lb, ub)
    best = max(X, key=f)

    for t in range(it):
        c1 = 2 * np.exp(-(4 * t / it) ** 2)

        for i in range(n):
            if i == 0:
                # Leader
                c2 = np.random.rand(d)

                if np.random.rand() < 0.5:
                    X[i] = best + c1 * ((ub - lb) * c2 + lb)
                else:
                    X[i] = best - c1 * ((ub - lb) * c2 + lb)
            else:
                # Followers
                X[i] = (X[i] + X[i - 1]) / 2

            X[i] = np.clip(X[i], lb, ub)

        current_best = max(X, key=f)
        if f(current_best) > f(best):
            best = current_best

    return best

def STO(f, d, it=100, n=30, lb=0, ub=1):
    X = init_pop1(n, d, lb, ub)
    best = max(X, key=f)

    for t in range(it):
        a = 2 * (1 - t / it)

        for i in range(n):
            r1 = np.random.rand(d)
            r2 = np.random.rand(d)

            if np.random.rand() < 0.5:
                # Exploration
                X_rand = X[np.random.randint(n)]
                X[i] = X[i] + r1 * (X_rand - r2 * X[i])
            else:
                # Exploitation
                X[i] = best + a * (r1 * (best - r2 * X[i]))

            X[i] = np.clip(X[i], lb, ub)

        current_best = max(X, key=f)
        if f(current_best) > f(best):
            best = current_best

    return best

def NMRA(f, d, it=100, n=30, lb=0, ub=1):
    X = init_pop1(n, d, lb, ub)
    best = max(X, key=f)

    for t in range(it):
        beta = 1 - (t / it)  # decay factor

        for i in range(n):
            r1 = np.random.rand(d)
            r2 = np.random.rand(d)

            # Nonlinear mutation (điểm khác biệt chính)
            mutation = beta * np.sin(2 * np.pi * r1) * (best - X[i])

            if np.random.rand() < 0.5:
                # Exploration
                X_rand = X[np.random.randint(n)]
                X[i] = X[i] + r2 * (X_rand - X[i]) + mutation
            else:
                # Exploitation
                X[i] = best + mutation

            # Boundary control
            X[i] = np.clip(X[i], lb, ub)

        # Update best
        current_best = max(X, key=f)
        if f(current_best) > f(best):
            best = current_best

    return best

algos = {
    "GA": GA, "Squirrel": Squirrel, "GWO": GWO,
    "MRA": MRA, "PSO": PSO, "SSA": SSA, "BBOA": BBOA,
    "ZOA": ZOA, "SSO": SSO, "STO": STO, "NMRA": NMRA
}


# ==========================
# Models
# ==========================
def get_models(input_dim):
    return {
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "LightGBM": LGBMClassifier(n_estimators=200, 
                                   learning_rate=0.05,verbose=-1,random_state=42),
        "CatBoost": CatBoostClassifier(iterations=200, verbose=0),
        "HGBoost": HistGradientBoostingClassifier(max_iter=200),
        "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, early_stopping=True),
        "ELM": ELM(input_dim)
    }


# ==========================
# Evaluation
# ==========================
def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    try:
        y_prob = model.predict_proba(X_test)
        if len(y_prob.shape) > 1:
            y_prob = y_prob[:,1]
    except:
        try:
            y_prob = model.decision_function(X_test)
        except:
            y_prob = model.predict(X_test)

    y_pred = model.predict(X_test)

    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "AUC": auc
    }


# ==========================
# MAIN
# ==========================
def run_all(folder=r"E:\DEU\DUE Scopus\NASA-promise-dataset-repository-main"):
    results = []
    all_top_feature_rows = []

    for file in sorted(os.listdir(folder)):
        if not file.endswith(".csv"): continue
        print("\nProcessing:", file)

        try:
            X, y = load_data(os.path.join(folder, file))

            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)

            unique, counts = np.unique(y, return_counts=True)

            if np.min(counts) < 2:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42
                )
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.3, random_state=42, stratify=y
                )

            d = X.shape[1]

            for algo_name, algo in algos.items():
                fit_fn = fitness_fs if algo_name in ("BBOA", "ZOA") else fitness
                sol = algo(lambda s: fit_fn(X_train, y_train, s), d)
                idx = np.argsort(sol)[-6:]

                for rank, feature_idx in enumerate(idx[::-1], 1):
                    all_top_feature_rows.append({
                        "Dataset": file,
                        "Algo": algo_name,
                        "Feature": f"F{feature_idx}",
                        "Rank": rank
                    })

                Xtr = X_train[:, idx]
                Xte = X_test[:, idx]

                models = get_models(len(idx))

                for model_name, model in models.items():
                    metrics = evaluate(model, Xtr, Xte, y_train, y_test)

                    res = {
                        "Dataset": file,
                        "Algo": algo_name,
                        "Model": model_name,
                        **{k: round(v,4) for k,v in metrics.items()}
                    }

                    results.append(res)
                    print(res)

        except Exception as e:
            print("Error:", file, e)

    df = pd.DataFrame(results)
    feat_df = pd.DataFrame(all_top_feature_rows)
    df.to_excel("SFP_results_nase6.xlsx", index=False)

    print("\n=== SUMMARY ===")
    #print(df.groupby(["Algo","Model"])["F1"].mean())
    summary = df.groupby(["Algo","Model"])[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].mean()
    print(summary)
    summary.to_excel('SFP_summary_nasa6.xlsx')

    return df,feat_df
# ==========================
# RUN
# ==========================
run_all()


Processing: cm1.csv
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'XGBoost', 'Accuracy': 0.8, 'Precision': 0.8, 'Recall': 0.8, 'F1': 0.8, 'AUC': 0.5721}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'LightGBM', 'Accuracy': 0.8, 'Precision': 0.8111, 'Recall': 0.8, 'F1': 0.8055, 'AUC': 0.6435}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'CatBoost', 'Accuracy': 0.9, 'Precision': 0.81, 'Recall': 0.9, 'F1': 0.8526, 'AUC': 0.562}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'HGBoost', 'Accuracy': 0.8067, 'Precision': 0.8225, 'Recall': 0.8067, 'F1': 0.8144, 'AUC': 0.599}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'MLP', 'Accuracy': 0.9, 'Precision': 0.81, 'Recall': 0.9, 'F1': 0.8526, 'AUC': 0.518}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'ELM', 'Accuracy': 0.8667, 'Precision': 0.8069, 'Recall': 0.8667, 'F1': 0.8357, 'AUC': 0.6074}
{'Dataset': 'cm1.csv', 'Algo': 'Squirrel', 'Model': 'XGBoost', 'Accuracy': 0.8533, 'Precision': 0.834, 'Recall': 0.8533, 'F1': 0.8431, 'AUC': 0.5963}
{'

(     Dataset  Algo     Model  Accuracy  Precision  Recall      F1     AUC
 0    cm1.csv    GA   XGBoost    0.8000     0.8000  0.8000  0.8000  0.5721
 1    cm1.csv    GA  LightGBM    0.8000     0.8111  0.8000  0.8055  0.6435
 2    cm1.csv    GA  CatBoost    0.9000     0.8100  0.9000  0.8526  0.5620
 3    cm1.csv    GA   HGBoost    0.8067     0.8225  0.8067  0.8144  0.5990
 4    cm1.csv    GA       MLP    0.9000     0.8100  0.9000  0.8526  0.5180
 ..       ...   ...       ...       ...        ...     ...     ...     ...
 325  pc1.csv  NMRA  LightGBM    0.9069     0.8815  0.9069  0.8931  0.7174
 326  pc1.csv  NMRA  CatBoost    0.9249     0.8995  0.9249  0.9076  0.8126
 327  pc1.csv  NMRA   HGBoost    0.9159     0.8919  0.9159  0.9019  0.7397
 328  pc1.csv  NMRA       MLP    0.0691     0.0048  0.0691  0.0089  0.3971
 329  pc1.csv  NMRA       ELM    0.9249     0.8942  0.9249  0.9039  0.7355
 
 [330 rows x 8 columns],
      Dataset  Algo Feature  Rank
 0    cm1.csv    GA     F10     1
 1   

In [27]:
from pathlib import Path

def export_top_features_by_fs_algorithm(feat_df, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    exports = []

    for fs_algo in feat_df["Algo"].unique():
        sub = feat_df[feat_df["Algo"] == fs_algo]

        # ===== bảng đầy đủ =====
        long_path = out_dir / f"{fs_algo}_top_features.csv"
        sub.to_csv(long_path, index=False)

        # ===== overlap (feature xuất hiện nhiều nhất) =====
        overlap = (
            sub["Feature"]
            .value_counts()
            .reset_index()
        )
        overlap.columns = ["Feature", "Count"]

        overlap_path = out_dir / f"{fs_algo}_feature_overlap.csv"
        overlap.to_csv(overlap_path, index=False)

        exports.append((fs_algo, long_path, overlap_path))

    return exports

In [28]:
from pathlib import Path

def ensure_outputs_dir(base_dir="outputs"):
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def export_comparison_table(df, out_dir):
    csv_path = out_dir / "comparison_results.csv"
    df.to_csv(csv_path, index=False)

    print(f"Saved CSV: {csv_path}")
    return csv_path

In [29]:
DATASETS: list[tuple[str, str]] = [
    ("CM1", "cm1"),
    ("KC1", "kc1"),
    ("KC2", "kc2"),
    ("JM1", "jm1"),
    ("PC1", "pc1"),
]

CLASSIFIER_SHORT = {
    "RandomForest": "RF",
    "RandomForestClassifier": "RF",
    "SVC": "SVM",
    "SupportVectorMachine": "SVM",
    "KNeighborsClassifier": "KNN",
    "LogisticRegression": "LR",
    "DecisionTreeClassifier": "DT",
    "NaiveBayes": "NB",
    "GaussianNB": "NB",
    "XGBClassifier": "XGB",
    "CatBoostClassifier": "CAT"
}

OPTIMIZERS: list[tuple[str, str]] = [
    ("GA", "Genetic Algorithm"),
    ("PSO", "Particle Swarm Optimization"),
    ("GWO", "Grey Wolf Optimizer"),
    ("Squirrel", "Squirrel Search Algorithm"),
    ("MRA", "Multiverse Random Algorithm"),
    ("SSA", "Salp Swarm Algorithm"),
    ("BBOA", "OriginalBBOA"),
    ("ZOA", "Zebra Optimization Algorithm"),
    ("SSO", "Salp Swarm Optimization"),
    ("STO", "Siberian Tiger Optimization"),
    ("NMRA", "NMRA Enhanced Algorithm"),
]


_RENAME_LOWER = True

In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

def export_comparison_charts(df, out_dir):
    """Grouped bar grid (P/R/F1/AUC per dataset) + F1 heatmap."""
    df = df.copy()

    df["classifier_short"] = df["Model"].map(
        lambda c: CLASSIFIER_SHORT.get(c, c[:12])
    )

    dataset_order = [d.lower() + ".csv" for d, _ in DATASETS]

    metrics = [
        ("Precision", "Precision"),
        ("Recall", "Recall"),
        ("F1", "F1"),
        ("AUC", "AUC"),
    ]

    n_rows, n_cols = len(dataset_order), len(metrics)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5.5 * n_cols, 4.5 * n_rows),
        constrained_layout=True,
    )

    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes.reshape(1, -1)

    palette = sns.color_palette("husl", n_colors=len(OPTIMIZERS))
    hue_order = [fs for fs, _ in OPTIMIZERS]

    for ri, ds in enumerate(dataset_order):
        sub = df[df["Dataset"] == ds]

        if sub.empty:
            continue   # ✅ tránh lỗi

        for ci, (col, title) in enumerate(metrics):
            ax = axes[ri, ci]

            sns.barplot(
                data=sub,
                x="classifier_short",
                y=col,
                hue="Algo",
                hue_order=hue_order,
                palette=palette,
                ax=ax,
            )

            ax.set_xlabel("")
            ax.set_ylabel(title)
            ax.set_title(f"{ds} — {title}")

            # Chỉ giữ legend ở subplot đầu tiên
            if not (ri == 0 and ci == 0):
                leg = ax.get_legend()
                if leg:
                    leg.remove()

    # Legend chung
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, title="FS", loc="upper right")

    grid_path = out_dir / "comparison_metrics_grid.png"

    fig.suptitle(
        "So sánh P / R / F1 / AUC (test) — thuật toán FS [NASA PROMISE]",
        fontsize=12,
    )

    fig.savefig(grid_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    # ===== HEATMAP =====
    plt.figure()

    heatmap_data = df.groupby(["Algo", "Model"])["F1"].mean().unstack()

    sns.heatmap(heatmap_data, annot=True)

    plt.title("F1 Heatmap")
    plt.tight_layout()

    heatmap_path = out_dir / "f1_heatmap.png"
    plt.savefig(heatmap_path)
    plt.close()

    print(f"Saved charts: {grid_path}, {heatmap_path}")

    return grid_path, heatmap_path

In [31]:
# ==========================
# EXPORT ALL
# ==========================
from pathlib import Path

OUTPUTS_DIR = Path("outputs/2")
TOP_FEATURES_DIR = OUTPUTS_DIR / "top_features"
N_FEATURES_EVAL = 6

df,feat_df = run_all()

# ===== EXPORT =====
out_dir = ensure_outputs_dir(OUTPUTS_DIR)

csv_path = export_comparison_table(df, out_dir)
grid_png, heatmap_png = export_comparison_charts(df, out_dir)

top_exports = export_top_features_by_fs_algorithm(feat_df, TOP_FEATURES_DIR)

# ===== PRINT =====
print("\n========== Đã export vào outputs/nasa/ ==========\n")
print(f"  Bảng CSV:       {csv_path}")
print(f"  Biểu đồ lưới:  {grid_png}")
print(f"  Heatmap F1:    {heatmap_png}")
print(f"  Top-{N_FEATURES_EVAL} features theo thuật toán FS → {TOP_FEATURES_DIR}/")

for fs_label, long_p, overlap_p in top_exports:
    print(f"    [{fs_label}]")
    print(f"      {long_p}")
    print(f"      {overlap_p}")


Processing: cm1.csv
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'XGBoost', 'Accuracy': 0.9, 'Precision': 0.875, 'Recall': 0.9, 'F1': 0.8802, 'AUC': 0.523}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'LightGBM', 'Accuracy': 0.86, 'Precision': 0.8467, 'Recall': 0.86, 'F1': 0.853, 'AUC': 0.6099}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'CatBoost', 'Accuracy': 0.8933, 'Precision': 0.8094, 'Recall': 0.8933, 'F1': 0.8493, 'AUC': 0.5881}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'HGBoost', 'Accuracy': 0.88, 'Precision': 0.8468, 'Recall': 0.88, 'F1': 0.8599, 'AUC': 0.5659}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'MLP', 'Accuracy': 0.9, 'Precision': 0.81, 'Recall': 0.9, 'F1': 0.8526, 'AUC': 0.4588}
{'Dataset': 'cm1.csv', 'Algo': 'GA', 'Model': 'ELM', 'Accuracy': 0.88, 'Precision': 0.8082, 'Recall': 0.88, 'F1': 0.8426, 'AUC': 0.6163}
{'Dataset': 'cm1.csv', 'Algo': 'Squirrel', 'Model': 'XGBoost', 'Accuracy': 0.8533, 'Precision': 0.8217, 'Recall': 0.8533, 'F1': 0.8366, 'AUC': 0